# Users overview: what we know about RentFriends users

Scope: the `users` table (one row per profile) and the user-related tables that reference it
(`profile_history`, `swipes`, `matches`, `chats`). Data is read from `data/processed/`, which is
local-only and gitignored.

Privacy: the `bio` text is **not displayed** here. It may contain names, phone numbers or social
links, and the data audit is still pending (see `docs/RENTFRIENDS.md`, section 4). Only `bio_length`
and a flag for whether a bio exists are shown.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", None)

# One hue for magnitude; a lighter tint only for a second series. Grid stays recessive.
PRIMARY = "#2F5D8C"
LIGHT = "#9DB8D6"
MUTED = "#8A8A8A"
plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.6,
        "figure.dpi": 110,
    }
)

DATA_DIR = Path("../data/processed")
users = pd.read_parquet(DATA_DIR / "users.parquet")
profile_history = pd.read_parquet(DATA_DIR / "profile_history.parquet")
swipes = pd.read_parquet(DATA_DIR / "swipes.parquet")
matches = pd.read_parquet(DATA_DIR / "matches.parquet")
chats = pd.read_parquet(DATA_DIR / "chats.parquet")

print(f"users: {len(users)} rows, {users['user_key'].nunique()} unique user_key")

## 1. Tables that carry user information

Keys are pseudonymised UUIDs (`user_key`). Every `user_key` in `swipes` and `profile_history` is
expected to exist in `users`; this is checked in section 6.

In [ ]:
overview = pd.DataFrame(
    {
        "table": ["users", "profile_history", "swipes", "matches", "chats"],
        "rows": [len(users), len(profile_history), len(swipes), len(matches), len(chats)],
        "user columns": [
            "user_key",
            "user_key",
            "user_key (actor), target_user_key (if target_type = profile)",
            "user_a_key, user_b_key",
            "user_a_key, user_b_key, first_sender_key",
        ],
    }
)
overview

## 2. Fields in `users`

Field meanings come from the source metadata (`data/tables_metadata.json`) and the exploration card
(`docs/rentfriends_hipotezy_karta_eksploracji.md`).

| Field | Meaning | Note |
|---|---|---|
| `user_key` | pseudonymised user id | unique in this table |
| `sex` | F / M / O | 73 users have `O` |
| `age_band` | age bracket, not years | 9 missing |
| `city` | city; `inne` = other | includes `Hyderabad` (45 users), which is unexpected for a Polish app |
| `room_type` | `SN`, `SH` (codes) | meaning not in repo, to confirm with the provider |
| `search_goal` | `LR`, `HA` (codes) | meaning not in repo, to confirm with the provider |
| `daily_routine`, `lifestyle`, `work_from_home`, `cleanliness`, `noise_level`, `smoking`, `alcohol` | lifestyle codes | **no codebook in repo**; codes are shown as they are |
| `interests` | JSON list of tags (0 to 9) | see section 4 |
| `budget` | monthly budget | stored as decimal; see section 3 |
| `move_in_month_offset` | months until move-in | includes negative values; see section 3 |
| `bio`, `bio_length` | free-text bio and its length | text not shown |

## 3. Missingness and data types

In [ ]:
summary = pd.DataFrame(
    {
        "dtype": users.dtypes.astype(str),
        "missing": users.isna().sum(),
        "missing %": (users.isna().mean() * 100).round(1),
        "distinct (non-null)": users.nunique(dropna=True),
    }
)
summary

In [ ]:
missing_pct = (users.isna().mean() * 100).drop("user_key").sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(missing_pct.index, missing_pct.values, color=PRIMARY)
for y, value in enumerate(missing_pct.values):
    ax.text(value + 1, y, f"{value:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 100)
ax.set_xlabel("missing (% of users)")
ax.set_title("Share of missing values per field in users")
plt.tight_layout()
plt.show()

## 4. Value sets of categorical fields

Counts include missing values (`NaN`). Codes are shown exactly as stored in the data.

In [ ]:
CATEGORICAL = [
    "sex",
    "age_band",
    "city",
    "room_type",
    "search_goal",
    "daily_routine",
    "lifestyle",
    "work_from_home",
    "cleanliness",
    "noise_level",
    "smoking",
    "alcohol",
]

for col in CATEGORICAL:
    counts = users[col].value_counts(dropna=False)
    table = pd.DataFrame({"n": counts, "%": (counts / len(users) * 100).round(1)})
    print(f"\n### {col} ({len(counts)} values incl. NaN)")
    print(table.to_string())

In [ ]:
# Small multiples: one panel per field, one axis each. Missing values are shown in muted gray.
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
for ax, col in zip(axes.flat, CATEGORICAL, strict=False):
    counts = users[col].value_counts(dropna=False)
    labels = ["NaN" if pd.isna(v) else str(v) for v in counts.index]
    colors = [MUTED if pd.isna(v) else PRIMARY for v in counts.index]
    ax.barh(labels, counts.values / len(users) * 100, color=colors)
    ax.invert_yaxis()
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("% of users", fontsize=8)
    ax.tick_params(labelsize=8)
fig.suptitle("Value sets of categorical fields (gray = missing)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Numeric fields and anomalies

- `move_in_month_offset` has two negative values (-222, -338). They are probably entry errors, not
  real offsets. They are kept here and flagged; the analysis decision is pending.
- `budget` is present for only 290 users. Most values fall between 1000 and 3000. Seven values
  lie outside 500 to 5000 (2, 10, 8000, 10000, 15000, 30000, 30000). Likely outliers or entry
  errors, to check with the provider.
- `budget` is stored as `Decimal` (object dtype) and is converted to float only for the summary.

In [ ]:
print("move_in_month_offset (value: n)")
print(users["move_in_month_offset"].value_counts(dropna=False).sort_index().to_string())

budget = pd.to_numeric(users["budget"])
print("\nbudget summary")
print(budget.describe().round(1).to_string())
print(
    "\nbudget values above 5000 or below 500:",
    sorted(budget[(budget > 5000) | (budget < 500)].dropna().tolist()),
)

In [ ]:
offsets = users["move_in_month_offset"].dropna().astype(int).value_counts().sort_index()
budget_in_range = budget[budget.between(0, 5000)].dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.bar(offsets.index.astype(str), offsets.values, color=PRIMARY)
ax1.set_title("move_in_month_offset (non-missing)")
ax1.set_xlabel("months (negative values are suspicious)")
ax1.set_ylabel("users")
ax2.hist(budget_in_range, bins=50, color=PRIMARY)
ax2.set_title("budget, values 0 to 5000 (outliers above 5000 not shown)")
ax2.set_xlabel("budget")
ax2.set_ylabel("users")
plt.tight_layout()
plt.show()

## 6. Interest tags (`interests`)

Each user has a JSON list of tags. The tag set has two styles that appear to describe the same
concepts:

- **snake_case keys**, for example `student`, `likes_calm`, `non_smoker`;
- **emoji plus label**, in Polish or English, for example `🎓 Student`, `🤫 Cisza po 22`.

Whether these pairs are the same tag has not been verified. Counts below are for the raw strings.

In [ ]:
tag_lists = users["interests"].apply(json.loads)
tag_counts = pd.Series([tag for tags in tag_lists for tag in tags]).value_counts()
n_tags = tag_lists.str.len()

is_snake_case = tag_counts.index.str.fullmatch(r"[a-z_]+")
print(f"distinct raw tags: {len(tag_counts)}")
print(f"  snake_case: {is_snake_case.sum()}")
print(f"  emoji / label style: {(~is_snake_case).sum()}")
print(f"\ntags per user: min {n_tags.min()}, median {n_tags.median()}, max {n_tags.max()}")
print(f"users at the cap of 9 tags: {(n_tags == 9).sum()} of {len(users)}")
print(f"users with no tags: {(n_tags == 0).sum()}")

print("\nAll tags (raw string: number of users)")
print(tag_counts.to_string())

In [ ]:
top = tag_counts.head(30)[::-1]
is_snake_case_top = top.index.str.fullmatch(r"[a-z_]+")
colors = [PRIMARY if snake else LIGHT for snake in is_snake_case_top]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top.index, top.values, color=colors)
ax.set_xlabel("users")
ax.set_title("Top 30 raw interest tags")
ax.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color=PRIMARY, label="snake_case key"),
        plt.Rectangle((0, 0), 1, 1, color=LIGHT, label="emoji / label"),
    ],
    loc="lower right",
)
plt.tight_layout()
plt.show()

tags_per_user = n_tags.value_counts().reindex(range(10), fill_value=0)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(tags_per_user.index.astype(str), tags_per_user.values, color=PRIMARY)
ax.set_xlabel("tags per user")
ax.set_ylabel("users")
ax.set_title("Number of interest tags per user (9 is the cap)")
plt.tight_layout()
plt.show()

## 7. Activity per user

Activity is counted from the action tables. A match counts once for each of its two users.

In [ ]:
activity = pd.DataFrame(index=users["user_key"])
activity["profile_versions"] = profile_history.groupby("user_key").size()
activity["swipes_made"] = swipes.groupby("user_key").size()
activity["right_swipes_made"] = swipes[swipes["direction"] == "right"].groupby("user_key").size()
activity["matches"] = pd.concat([matches["user_a_key"], matches["user_b_key"]]).value_counts()
activity["chats"] = pd.concat([chats["user_a_key"], chats["user_b_key"]]).value_counts()
activity = activity.fillna(0).astype(int)

print(activity.describe().round(1).to_string())
print("\nusers with zero activity in each column:")
print((activity == 0).sum().to_string())

print("\nreferential check (keys in swipes/profile_history that are not in users):")
print("  swipes.user_key:", (~swipes["user_key"].isin(users["user_key"])).sum())
print("  profile_history.user_key:", (~profile_history["user_key"].isin(users["user_key"])).sum())

In [ ]:
# Activity counts have a long tail, so use log-scaled bins; only users with at least one event.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["swipes_made", "right_swipes_made", "chats"], strict=False):
    values = activity.loc[activity[col] > 0, col]
    bins = np.logspace(0, np.log10(values.max()), 40)
    ax.hist(values, bins=bins, color=PRIMARY)
    ax.set_xscale("log")
    ax.set_title(f"{col} (users with at least 1)")
    ax.set_xlabel("count (log scale)")
    ax.set_ylabel("users")
plt.tight_layout()
plt.show()

## 8. Two random users

Sample with `random_state=42` for reproducibility. Bio text is replaced by a flag.

In [ ]:
sample = users.sample(n=2, random_state=42)

view = sample.drop(columns=["bio"]).set_index("user_key")
view["interests"] = sample.set_index("user_key")["interests"].apply(
    lambda s: ", ".join(json.loads(s))
)
view["has_bio"] = (sample.set_index("user_key")["bio_length"] > 0).values
view = pd.concat([view, activity.loc[view.index]], axis=1)
view.T

## 9. Open points

- The lifestyle and room codes (`LR`, `HA`, `SN`, `SH`, `V`, `NO`, `FT`, ...) need a codebook from
  the provider before they are interpreted.
- The `Hyderabad` entries, the negative `move_in_month_offset` values and the extreme `budget`
  values need checking.
- The duplicate tag styles need a mapping before tag-based similarity (Jaccard) is computed.
- Bio content needs the personal-data audit before any text is displayed or analysed.